# Evaluating Model Outputs

We can evaluate a model's confidence in its results by using perplexity. Perplexity is a measure of uncertainty that can be calculated by exponentiating the negative of the average of the logprobs. 

+ Perplexity can be used to assess the result of an individual model run.
+ It can also be used to compare the relative confidence of results between model runs. 

Low perplexity or high confidence does not guarantee accuracy, but it can be a helpful signal when paired with other evaluation metrics. 

In [ ]:
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets
import sys
sys.path.append('../../05_src/')

In [ ]:
from utils.clients import get_client
from IPython.display import display, Markdown
import numpy as np

client = get_client()

In [ ]:
prompts = [
    # Low perplexity: Clear topic, common structure, highly predictable vocabulary
    "Explain how photosynthesis works in simple terms.",
    # Medium preplexity: Narrative freedom, but familiar theme and constraints.
    "Write a short story about a traveler who realizes the journey mattered more than the destination.",
    # High perplexity: Abstract concept, creative freedom, unpredictable vocabulary
    "Describe the taste of a color that only exists for one second at dusk, using metaphors from mathematics and weather."
]

In [ ]:
def get_completion(
    input: list[dict[str, str]],
    model: str = "gpt-4o-mini",
    max_tokens=500,
    temperature=0,
    tools=None,
    logprobs=None,  # whether to return log probabilities of the output tokens or not. If true, returns the log probabilities of each output token returned in the content of message..
    top_logprobs=None,
) -> str:
    params = {
        "model": model,
        "input": input,
        "max_output_tokens": max_tokens,
        "temperature": temperature,
        "tools": tools,
        "include": ["message.output_text.logprobs"] if logprobs else [],
        "top_logprobs": top_logprobs,
    }
    if tools:
        params["tools"] = tools

    completion = client.responses.create(**params)
    return completion

In [ ]:
for k, prompt in enumerate(prompts):
    API_RESPONSE = get_completion(
        [{"role": "user", "content": prompt}],
        model="gpt-4o-mini",
        logprobs=True,
    )
    token_data = API_RESPONSE.output[0].content[0].logprobs
    response_text = API_RESPONSE.output[0].content[0].text
    lp_values = [t.logprob for t in token_data]
    perplexity_score = np.exp(-np.mean(lp_values))

    rows = [
        f"| `{t.token.replace('|', chr(9474)).replace(chr(10), '↵').replace(chr(96), chr(39))}` "
        f"| {t.logprob:.4f} | {np.exp(t.logprob)*100:.1f}% |"
        for t in token_data
    ]
    table = "\n".join([
        "| Token | Logprob | Linear Prob |",
        "|:------|--------:|------------:|",
    ] + rows)

    display(Markdown(f"### Prompt {k+1}\n_{prompt}_"))
    display(Markdown(f"**Response:**\n\n{response_text}"))
    display(Markdown(table))
    display(Markdown(f"**Perplexity:** `{perplexity_score:.2f}`\n\n---"))